# Experiment 2: open briefs
Two new AI sets, 30 images each:

* **ai_category**: each image gets only the *type* of your matching design (e.g. "A handmade gift basket"), so the AI chooses everything else.
* **ai_open**: every image gets the *same* open brief, with a different seed.

Run on a Colab **GPU** runtime (T4). Total time is about 25-30 minutes.

In [ ]:
!pip install -q diffusers transformers accelerate

In [ ]:
import torch
assert torch.cuda.is_available(), 'Switch the runtime to a GPU first'
print(torch.cuda.get_device_name(0))

In [ ]:
CATEGORY_PROMPTS = [
    ('01', 'A handmade gift basket, product photograph'),
    ('02', 'A gift arrangement in a clear acrylic box, product photograph'),
    ('03', 'A handmade gift basket, product photograph'),
    ('04', 'A handmade gift basket, product photograph'),
    ('05', 'A handmade gift bouquet, product photograph'),
    ('06', 'A handmade gift basket, product photograph'),
    ('07', 'A set of three small handmade gift bouquets, product photograph'),
    ('08', 'A set of three small handmade gift bouquets, product photograph'),
    ('09', 'A set of three small handmade gift bouquets, product photograph'),
    ('10', 'A handmade gift bouquet, product photograph'),
    ('11', 'A handmade gift basket, product photograph'),
    ('12', 'A handmade gift box, product photograph'),
    ('13', 'A handmade gift basket, product photograph'),
    ('14', 'A handmade gift basket, product photograph'),
    ('15', 'A gift arrangement in a clear acrylic box, product photograph'),
    ('16', 'A pair of handmade gift boxes, product photograph'),
    ('17', 'A handmade gift basket, product photograph'),
    ('18', 'A jewelry gift, product photograph'),
    ('19', 'A handmade gift box, product photograph'),
    ('20', 'A handmade gift card, product photograph'),
    ('21', 'A handmade gift basket, product photograph'),
    ('22', 'A pair of printed gift cushions, product photograph'),
    ('23', 'A handmade gift bouquet, product photograph'),
    ('24', 'A handmade gift box, product photograph'),
    ('25', 'A handmade gift card, product photograph'),
    ('26', 'A handmade gift basket, product photograph'),
    ('27', 'A small gift bag arrangement, product photograph'),
    ('28', 'A handmade gift bouquet, product photograph'),
    ('29', 'A decorated celebration cake gift tray, product photograph'),
    ('30', 'A handmade gift basket, product photograph'),
]

OPEN_BRIEF = 'A handmade gift arrangement from a small gifting business, product photograph'


In [ ]:
from diffusers import StableDiffusionXLPipeline

MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
STEPS = 30
GUIDANCE = 7.0

pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, variant='fp16', use_safetensors=True).to('cuda')

In [ ]:
import os

def make(prompt, seed, path):
    g = torch.Generator('cuda').manual_seed(seed)
    pipe(prompt=prompt, num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=g).images[0].save(path)

os.makedirs('ai_category', exist_ok=True)
for pid, prompt in CATEGORY_PROMPTS:
    make(prompt, 3000 + int(pid), f'ai_category/ai{pid}.png')
    print('category', pid)

In [ ]:
os.makedirs('ai_open', exist_ok=True)
for pid, _ in CATEGORY_PROMPTS:
    make(OPEN_BRIEF, 2000 + int(pid), f'ai_open/ai{pid}.png')
    print('open', pid)

In [ ]:
import json
settings = {'model': MODEL_ID, 'steps': STEPS, 'guidance': GUIDANCE,
            'ai_category': {'seed_rule': '3000 + id', 'prompts': dict(CATEGORY_PROMPTS)},
            'ai_open': {'seed_rule': '2000 + id', 'prompt': OPEN_BRIEF}}
json.dump(settings, open('experiment2_settings.json', 'w'), indent=2)
!zip -qr experiment2.zip ai_category ai_open experiment2_settings.json
print('zipped')

## Download
Keep this tab open until the download finishes.

In [ ]:
from google.colab import files
files.download('experiment2.zip')